# SAM 3 multi-class pixel annotation for CMZ

This notebook preserves the existing civet annotations and adds a second pixel-level class, `food_bowl`. Both classes require human review. A bowl must be annotated whenever it is visible in Eating, Locomotion, and Resting frames; otherwise the model may incorrectly learn that the presence of a bowl alone means Eating.

## 1. Runtime, storage and SAM 3 installation

Colab: use a GPU runtime and add a Colab secret named `HF_TOKEN`. Local Windows: launch Jupyter from a CUDA-enabled Python environment, set `CMZ_PROJECT_ROOT`, and authenticate using `HF_TOKEN`, a cached Hugging Face login, or the secure interactive login prompt. The token is never written into this notebook.

In [ ]:
import sys, subprocess
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
SAM3_DIR = Path('/content/sam3')
SAM3_DIR.parent.mkdir(parents=True, exist_ok=True)
if not SAM3_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam3.git', str(SAM3_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{SAM3_DIR}[notebooks]'], check=True)
print('SAM 3:', SAM3_DIR)
print('If imports fail after the first installation, restart the kernel once and continue below.')

If imports fail after the first installation, restart the Colab session or local Jupyter kernel once. For local Windows usage, set the project path before starting Jupyter, for example: `$env:CMZ_PROJECT_ROOT='G:\My Drive\CMZprojectphase2'; jupyter notebook`. You may optionally set `$env:HF_TOKEN='hf_...'` for that PowerShell session; otherwise the notebook opens a secure login prompt.

In [ ]:
import sys, subprocess, zipfile, json, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

drive.mount('/content/drive')
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN was not found in Colab Secrets.')
login(token=hf_token)
PROJECT_ROOT = Path('/content/drive/MyDrive/CMZprojectphase2')
SAM3_DIR = Path('/content/sam3')
LOCAL_EXTRACT_ROOT = Path('/content/CMZ')
ZIP_PATH = PROJECT_ROOT / 'data' / 'cmz_processed_dataset.zip'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'sam3_batch_annotation'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Processed dataset ZIP not found: {ZIP_PATH}')
print('Project root:', PROJECT_ROOT)
print('SAM 3 directory:', SAM3_DIR)
print('Dataset ZIP:', ZIP_PATH)
print('Annotation output:', OUTPUT_ROOT)

## 2. Extract and inspect the friend's processed dataset

Extraction uses Colab local storage for speed. Outputs and review files are saved to Drive.

In [ ]:
manifest_candidates = list(LOCAL_EXTRACT_ROOT.rglob('metadata/manifest.csv')) if LOCAL_EXTRACT_ROOT.exists() else []
if not manifest_candidates:
    LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print('Extracting processed dataset to Colab local storage...')
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(LOCAL_EXTRACT_ROOT)
    manifest_candidates = list(LOCAL_EXTRACT_ROOT.rglob('metadata/manifest.csv'))

if len(manifest_candidates) != 1:
    raise RuntimeError(f'Expected one manifest.csv, found: {manifest_candidates}')
MANIFEST_PATH = manifest_candidates[0]
DATASET_BASE = MANIFEST_PATH.parent.parent
manifest = pd.read_csv(MANIFEST_PATH, low_memory=False)

required = {'video_id', 'label', 'lighting', 'timestamp_sec', 'full_path'}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f'Manifest is missing columns: {sorted(missing)}')

print('Dataset base:', DATASET_BASE)
print('Manifest rows:', len(manifest))
print('Videos:', manifest.video_id.nunique())
display(pd.crosstab(manifest['label'], manifest['lighting'], margins=True))

In [ ]:
sample_row = manifest[manifest['full_path'].notna()].iloc[0]
sample_path = DATASET_BASE / str(sample_row['full_path'])
print('Sample:', sample_path, 'exists:', sample_path.exists())
sample_image = Image.open(sample_path).convert('RGB')
print('Size:', sample_image.size)
display(sample_image)

## 3. Build a reproducible, video-diverse annotation selection

Frames are thinned to one candidate every two seconds, capped per video, and sampled within each available behavior-lighting group. Detector misses are included as hard-review candidates because a missing MegaDetector box does not prove that no civet is present.

The first pilot targets roughly 300-500 frames. Missing combinations such as night Eating are reported rather than fabricated.

In [ ]:
RANDOM_SEED = 42
SECONDS_BETWEEN_CANDIDATES = 2
TARGET_PER_LABEL_LIGHTING = 70
MAX_PER_VIDEO_PER_GROUP = 25
TARGET_DETECTOR_MISSES = 50
SELECTION_PATH = OUTPUT_ROOT / 'annotation_selection.csv'

def bool_series(frame, name, default=True):
    if name not in frame.columns:
        return pd.Series(default, index=frame.index)
    s = frame[name]
    if s.dtype == bool:
        return s.fillna(False)
    return s.astype(str).str.lower().isin(['true', '1', 'yes'])

if SELECTION_PATH.exists():
    selection = pd.read_csv(SELECTION_PATH)
    print('Reusing saved selection:', SELECTION_PATH)
else:
    candidates = manifest.copy()
    candidates = candidates[bool_series(candidates, 'readable', True)]
    candidates = candidates[candidates['label'].isin(['Eating', 'Locomotion', 'Resting'])]
    candidates = candidates[candidates['full_path'].notna() & candidates['video_id'].notna()]
    candidates['_time_bin'] = (pd.to_numeric(candidates['timestamp_sec'], errors='coerce') // SECONDS_BETWEEN_CANDIDATES).astype('Int64')
    candidates = candidates.dropna(subset=['_time_bin']).sort_values(['video_id', 'timestamp_sec'])
    candidates = candidates.drop_duplicates(['video_id', '_time_bin'])

    if 'box_x1' in candidates.columns:
        has_box = candidates['box_x1'].notna()
    elif 'civet_detected' in candidates.columns:
        has_box = bool_series(candidates, 'civet_detected', False)
    else:
        has_box = pd.Series(True, index=candidates.index)
    candidates['_has_prior_box'] = has_box

    chosen = []
    positives = candidates[candidates['_has_prior_box']].copy()
    for (label, lighting), group in positives.groupby(['label', 'lighting'], dropna=False):
        group = group.sample(frac=1, random_state=RANDOM_SEED)
        group = group.groupby('video_id', group_keys=False).head(MAX_PER_VIDEO_PER_GROUP)
        group = group.sample(min(TARGET_PER_LABEL_LIGHTING, len(group)), random_state=RANDOM_SEED)
        chosen.append(group.assign(annotation_kind='prior_detection'))

    misses = candidates[~candidates['_has_prior_box']].copy()
    if len(misses):
        misses = misses.sample(frac=1, random_state=RANDOM_SEED)
        misses = misses.groupby('video_id', group_keys=False).head(10)
        misses = misses.sample(min(TARGET_DETECTOR_MISSES, len(misses)), random_state=RANDOM_SEED)
        chosen.append(misses.assign(annotation_kind='detector_miss_review'))

    selection = pd.concat(chosen, ignore_index=True)
    selection.insert(0, 'selection_id', [f'A{i:05d}' for i in range(len(selection))])
    keep = ['selection_id', 'video_id', 'label', 'lighting', 'timestamp_sec', 'full_path', 'annotation_kind']
    for optional in ['split', 'fold', 'camera', 'window_id']:
        if optional in selection.columns:
            keep.append(optional)
    selection = selection[keep].sort_values(['label', 'lighting', 'video_id', 'timestamp_sec']).reset_index(drop=True)
    selection.to_csv(SELECTION_PATH, index=False)
    print('Saved selection:', SELECTION_PATH)

print('Selected frames:', len(selection))
display(pd.crosstab([selection['label'], selection['lighting']], selection['annotation_kind'], margins=True))
print('Frames per video (largest first):')
display(selection.video_id.value_counts().head(15).rename('frames').to_frame())

Inspect a random grid before running SAM 3. If the selection is dominated by near-duplicates or labels are visibly wrong, stop and correct the selection/manifest first.

In [ ]:
preview = selection.sample(min(24, len(selection)), random_state=RANDOM_SEED)
thumb_w, thumb_h, label_h, columns = 256, 256, 36, 6
rows = int(np.ceil(len(preview) / columns))
sheet = Image.new('RGB', (thumb_w*columns, (thumb_h+label_h)*rows), 'white')
draw = ImageDraw.Draw(sheet)
for i, row in enumerate(preview.itertuples()):
    path = DATASET_BASE / row.full_path
    image = Image.open(path).convert('RGB').resize((thumb_w, thumb_h))
    x, y = (i % columns)*thumb_w, (i // columns)*(thumb_h+label_h)
    sheet.paste(image, (x, y))
    draw.text((x+3, y+thumb_h+2), f'{row.selection_id} {row.label} {row.lighting}', fill='black')
display(sheet)

## 4. Load SAM 3

Paths are explicit to avoid tokenizer discovery errors. FP16 autocast is used for Tesla T4 inference; do not enable a global BF16 context.

In [ ]:
BPE_PATH = SAM3_DIR / 'sam3' / 'assets' / 'bpe_simple_vocab_16e6.txt.gz'
if not BPE_PATH.exists():
    raise FileNotFoundError(BPE_PATH)
CHECKPOINT_PATH = Path(hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt', token=hf_token))
if str(SAM3_DIR) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR))
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

gc.collect(); torch.cuda.empty_cache()
sam_model = build_sam3_image_model(
    bpe_path=str(BPE_PATH), checkpoint_path=str(CHECKPOINT_PATH),
    load_from_HF=False, device='cuda', eval_mode=True,
)
sam_processor = Sam3Processor(sam_model, device='cuda', confidence_threshold=0.5)
print('SAM 3 ready; dtype:', next(sam_model.parameters()).dtype)

## 5. Add resumable food-bowl proposals

Existing civet files remain unchanged (`mask.png`, `overlay.jpg`, `metadata.json`). This pass adds `mask_bowl.png`, `overlay_bowl.jpg`, and `bowl_metadata.json`. Every bowl proposal starts as `needs_review`.

In [ ]:
BOWL_PROMPTS = ['food bowl', 'feeding bowl', 'bowl', 'food tray']
BOWL_CONFIDENCE_THRESHOLD = 0.25
BATCH_LIMIT = 50
PROPOSAL_DIR = OUTPUT_ROOT / 'proposals'
PROPOSAL_DIR.mkdir(parents=True, exist_ok=True)

def to_numpy(value):
    return value.detach().float().cpu().numpy() if isinstance(value, torch.Tensor) else np.asarray(value)

def make_overlay(image, mask, color=(255, 165, 0), alpha=0.45):
    result = np.asarray(image).copy()
    result[mask] = ((1-alpha)*result[mask] + alpha*np.asarray(color)).astype(np.uint8)
    return Image.fromarray(result)

def run_prompt(image, prompt, threshold):
    sam_processor.set_confidence_threshold(threshold)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        state = sam_processor.set_image(image)
        return sam_processor.set_text_prompt(state=state, prompt=prompt)

def bowl_record_path(selection_id):
    return PROPOSAL_DIR / selection_id / 'bowl_metadata.json'

bowl_pending_mask = ~selection.selection_id.map(lambda x: bowl_record_path(x).exists())
bowl_pending = selection[bowl_pending_mask].head(BATCH_LIMIT)
print(f'Pending bowl proposals: {int(bowl_pending_mask.sum())}; processing now: {len(bowl_pending)}')

In [ ]:
from tqdm.auto import tqdm

for row in tqdm(list(bowl_pending.itertuples()), desc='SAM 3 bowl proposals'):
    item_dir = PROPOSAL_DIR / row.selection_id
    item_dir.mkdir(parents=True, exist_ok=True)
    image_path = item_dir / 'image.jpg'
    if image_path.exists():
        image = Image.open(image_path).convert('RGB')
    else:
        image = Image.open(DATASET_BASE / row.full_path).convert('RGB')
        image.save(image_path, quality=95)

    best_result = None
    for prompt in BOWL_PROMPTS:
        output = run_prompt(image, prompt, BOWL_CONFIDENCE_THRESHOLD)
        scores = to_numpy(output['scores']).reshape(-1)
        if len(scores) == 0:
            continue
        index = int(scores.argmax())
        score = float(scores[index])
        if best_result is None or score > best_result['score']:
            best_result = {'prompt': prompt, 'index': index, 'score': score, 'scores': scores, 'output': output}

    metadata = {
        'selection_id': row.selection_id, 'class_name': 'food_bowl',
        'video_id': row.video_id, 'timestamp_sec': float(row.timestamp_sec),
        'behavior': row.label, 'lighting': row.lighting,
        'source_full_path': row.full_path, 'threshold': BOWL_CONFIDENCE_THRESHOLD,
        'bowl_status': 'needs_review',
    }
    if best_result is None:
        image.save(item_dir / 'overlay_bowl.jpg', quality=95)
        metadata.update({'proposal_result': 'no_sam_detection', 'prompt': None, 'score': None, 'all_scores': []})
    else:
        output, index = best_result['output'], best_result['index']
        mask = np.squeeze(to_numpy(output['masks'])[index]).astype(bool)
        box = to_numpy(output['boxes'])[index].reshape(-1)[:4]
        Image.fromarray(mask.astype(np.uint8)*255).save(item_dir / 'mask_bowl.png')
        make_overlay(image, mask).save(item_dir / 'overlay_bowl.jpg', quality=95)
        metadata.update({
            'proposal_result': 'mask_proposed', 'prompt': best_result['prompt'],
            'selected_index': index, 'score': best_result['score'],
            'all_scores': [float(x) for x in best_result['scores']],
            'box_xyxy': [float(x) for x in box], 'mask_pixels': int(mask.sum()),
        })
    (item_dir / 'bowl_metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

gc.collect(); torch.cuda.empty_cache()
print('Bowl batch complete. Re-run the two bowl cells until pending is zero.')

## 6. Consolidate two-class review metadata and create contact sheets

`civet_status` is migrated from the existing `mask_status`. `bowl_status` is independent. Never mark a visible but missed object as negative/not-visible; correct it manually in CVAT instead.

In [ ]:
CONTACT_DIR = OUTPUT_ROOT / 'contact_sheets_multiclass'
CONTACT_DIR.mkdir(parents=True, exist_ok=True)
cols, rows, thumb, label_h = 5, 5, 220, 58
per_page = cols*rows
for page, start in enumerate(range(0, len(review), per_page), 1):
    chunk = review.iloc[start:start+per_page]
    sheet = Image.new('RGB', (cols*thumb, rows*(thumb+label_h)), 'white')
    draw = ImageDraw.Draw(sheet)
    for i, row in enumerate(chunk.itertuples()):
        item_dir = PROPOSAL_DIR / row.selection_id
        image = Image.open(item_dir / 'image.jpg').convert('RGB')
        combined = np.asarray(image).copy()
        for filename, color in [('mask.png', np.array([0,255,0])), ('mask_bowl.png', np.array([255,165,0]))]:
            path = item_dir / filename
            if path.exists():
                mask = np.asarray(Image.open(path).convert('L')) > 0
                combined[mask] = (0.55*combined[mask] + 0.45*color).astype(np.uint8)
        image = Image.fromarray(combined).resize((thumb, thumb))
        x, y = (i%cols)*thumb, (i//cols)*(thumb+label_h)
        sheet.paste(image, (x, y))
        draw.text((x+3, y+thumb+2), f'{row.selection_id} {row.behavior} {row.lighting}', fill='black')
        draw.text((x+3, y+thumb+20), f'civet={row.civet_status} bowl={row.bowl_status}', fill='black')
    sheet.save(CONTACT_DIR / f'review_{page:03d}.jpg', quality=90)
print('Green=civet; orange=food bowl. Contact sheets:', CONTACT_DIR)
if list(CONTACT_DIR.glob('review_*.jpg')):
    display(Image.open(sorted(CONTACT_DIR.glob('review_*.jpg'))[0]))

In [ ]:
print("สรุปผลการตรวจจับชาม")

display(
    review["bowl_result"]
    .value_counts(dropna=False)
    .rename("images")
    .to_frame()
)

bowl_proposed = review[
    review["bowl_result"] == "mask_proposed"
].copy()

bowl_missed = review[
    review["bowl_result"]
    == "no_sam_detection"
].copy()

print("มี proposal ชาม:", len(bowl_proposed))
print("SAM ไม่พบชาม:", len(bowl_missed))

display(
    review[
        [
            "selection_id",
            "behavior",
            "lighting",
            "bowl_result",
            "bowl_prompt",
            "bowl_score",
            "bowl_status",
        ]
    ]
)

In [ ]:
from tqdm.auto import tqdm
import json
import gc

BOWL_RETRY_THRESHOLD = 0.10
BOWL_RETRY_LIMIT = 50

BOWL_RETRY_PROMPTS = [
    "food bowl",
    "feeding bowl",
    "feeding dish",
    "food dish",
    "animal bowl",
    "plate",
    "food tray",
]

retry_candidates = []

for metadata_path in PROPOSAL_DIR.glob(
    "*/bowl_metadata.json"
):
    metadata = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    if (
        metadata.get("proposal_result")
        == "no_sam_detection"
        and not metadata.get(
            "bowl_retry_v1_done",
            False,
        )
    ):
        retry_candidates.append({
            "selection_id": metadata["selection_id"],
            "metadata_path": metadata_path,
        })

bowl_retry_pending = retry_candidates[
    :BOWL_RETRY_LIMIT
]

print(
    "ยังไม่ Retry:",
    len(retry_candidates),
)

print(
    "กำลังทำรอบนี้:",
    len(bowl_retry_pending),
)

In [ ]:
for item in tqdm(
    bowl_retry_pending,
    desc="Retry bowl masks",
):
    selection_id = item["selection_id"]
    metadata_path = item["metadata_path"]
    item_dir = metadata_path.parent

    image = Image.open(
        item_dir / "image.jpg"
    ).convert("RGB")

    best_result = None

    for prompt in BOWL_RETRY_PROMPTS:
        output = run_prompt(
            image,
            prompt,
            BOWL_RETRY_THRESHOLD,
        )

        scores = to_numpy(
            output["scores"]
        ).reshape(-1)

        if len(scores) == 0:
            continue

        best_index = int(scores.argmax())
        best_score = float(
            scores[best_index]
        )

        if (
            best_result is None
            or best_score > best_result["score"]
        ):
            best_result = {
                "prompt": prompt,
                "index": best_index,
                "score": best_score,
                "scores": scores,
                "output": output,
            }

    metadata = json.loads(
        metadata_path.read_text(
            encoding="utf-8"
        )
    )

    metadata["bowl_retry_v1_done"] = True
    metadata["retry_threshold"] = (
        BOWL_RETRY_THRESHOLD
    )

    if best_result is None:
        metadata["retry_result"] = (
            "still_no_detection"
        )

    else:
        output = best_result["output"]
        index = best_result["index"]

        mask = np.squeeze(
            to_numpy(
                output["masks"]
            )[index]
        ).astype(bool)

        box = to_numpy(
            output["boxes"]
        )[index].reshape(-1)[:4]

        Image.fromarray(
            mask.astype(np.uint8) * 255
        ).save(
            item_dir / "mask_bowl.png"
        )

        make_overlay(
            image,
            mask,
            color=(255, 165, 0),
        ).save(
            item_dir / "overlay_bowl.jpg",
            quality=95,
        )

        metadata.update({
            "proposal_result":
                "mask_proposed_retry",
            "retry_result":
                "mask_proposed",
            "prompt":
                best_result["prompt"],
            "selected_index": index,
            "score":
                best_result["score"],
            "all_scores": [
                float(x)
                for x
                in best_result["scores"]
            ],
            "box_xyxy": [
                float(x) for x in box
            ],
            "mask_pixels":
                int(mask.sum()),
            "bowl_status":
                "needs_review",
        })

    metadata_path.write_text(
        json.dumps(
            metadata,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

gc.collect()
torch.cuda.empty_cache()

print(
    "Retry batch complete — "
    "รันสองเซลล์นี้ซ้ำเพื่อทำชุดถัดไป"
)

In [ ]:
display(
    review["bowl_result"]
    .value_counts(dropna=False)
    .rename("images")
    .to_frame()
)

In [ ]:
records = []
for civet_path in PROPOSAL_DIR.glob('*/metadata.json'):
    civet = json.loads(civet_path.read_text(encoding='utf-8'))
    selection_id = str(civet['selection_id'])
    bowl_path = civet_path.parent / 'bowl_metadata.json'
    bowl = json.loads(bowl_path.read_text(encoding='utf-8')) if bowl_path.exists() else {}
    records.append({
        **civet,
        'civet_result': civet.get('proposal_result'),
        'civet_score': civet.get('score'),
        'civet_status': civet.get('civet_status', civet.get('mask_status', 'needs_review')),
        'bowl_result': bowl.get('proposal_result', 'not_processed'),
        'bowl_score': bowl.get('score'),
        'bowl_prompt': bowl.get('prompt'),
        'bowl_status': bowl.get('bowl_status', 'needs_review'),
    })

review = pd.DataFrame(records)
if review.empty:
    raise RuntimeError('No civet proposals exist yet.')
review = review.sort_values(['behavior', 'lighting', 'video_id', 'timestamp_sec'])
REVIEW_CSV = OUTPUT_ROOT / 'review_status.csv'
if REVIEW_CSV.exists():
    old = pd.read_csv(REVIEW_CSV)
    old = old.set_index('selection_id')
    for column in ['civet_status', 'bowl_status']:
        if column in old.columns:
            edited = old[column].to_dict()
            review[column] = review.apply(lambda r: edited.get(r.selection_id, r[column]), axis=1)
    if 'mask_status' in old.columns and 'civet_status' not in old.columns:
        edited = old['mask_status'].to_dict()
        review['civet_status'] = review.apply(lambda r: edited.get(r.selection_id, r.civet_status), axis=1)
review.to_csv(REVIEW_CSV, index=False)
print('Civet records:', len(review), '/', len(selection))
print('Bowl records:', int(review.bowl_result.ne('not_processed').sum()), '/', len(selection))
display(review[['civet_status', 'bowl_status']].value_counts(dropna=False).rename('images').to_frame())
print('Review CSV:', REVIEW_CSV)

### Interactive bowl review

Review every proposed bowl mask. Accept correct orange masks, confirm genuinely absent bowls, or keep unresolved images excluded from export.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

if 'bowl_needs_cvat' not in review.columns:
    review['bowl_needs_cvat'] = False

bowl_queue = review.index[review['bowl_status'].eq('needs_review')].tolist()
review_position = {'value': 0}

def save_bowl_decision(index, status, needs_cvat=False):
    review.at[index, 'bowl_status'] = status
    review.at[index, 'bowl_needs_cvat'] = needs_cvat
    review.to_csv(REVIEW_CSV, index=False)

def show_bowl_review():
    clear_output(wait=True)
    position = review_position['value']
    if position >= len(bowl_queue):
        print('Bowl review complete. Saved:', REVIEW_CSV)
        display(review['bowl_status'].value_counts(dropna=False).rename('images').to_frame())
        return
    index = bowl_queue[position]
    row = review.loc[index]
    item_dir = PROPOSAL_DIR / str(row.selection_id)
    original = Image.open(item_dir / 'image.jpg').convert('RGB')
    mask_path = item_dir / 'mask_bowl.png'
    overlay = np.asarray(original).copy()
    if mask_path.exists():
        mask = np.asarray(Image.open(mask_path).convert('L')) > 0
        overlay[mask] = (0.55*overlay[mask] + 0.45*np.array([255,165,0])).astype(np.uint8)
    comparison = Image.new('RGB', (original.width*2, original.height), 'white')
    comparison.paste(original, (0,0)); comparison.paste(Image.fromarray(overlay), (original.width,0))
    print(f'{position+1}/{len(bowl_queue)} {row.selection_id} | {row.behavior} | {row.lighting}')
    print('Left=original; right=orange bowl proposal')
    display(comparison.resize((comparison.width//2, comparison.height//2)))

    accept = widgets.Button(description='Accept mask', button_style='success', disabled=not mask_path.exists())
    absent = widgets.Button(description='No bowl', button_style='info')
    defer = widgets.Button(description='Keep excluded', button_style='warning')
    reject = widgets.Button(description='Reject image', button_style='danger')
    def decide(status, needs_cvat=False):
        save_bowl_decision(index, status, needs_cvat)
        review_position['value'] += 1
        show_bowl_review()
    accept.on_click(lambda _: decide('accepted'))
    absent.on_click(lambda _: decide('not_visible'))
    defer.on_click(lambda _: decide('needs_review', True))
    reject.on_click(lambda _: decide('rejected'))
    display(widgets.HBox([accept, absent, defer, reject]))

show_bowl_review()

## 7. Human review rules

Review both objects independently:

- `accepted`: the saved mask completely and correctly covers the visible object.
- `corrected`: a human corrected the mask in CVAT.
- `negative` (civet only): a human confirmed that no civet is visible.
- `not_visible` (bowl only): a human confirmed that no food bowl/tray is visible.
- `needs_review`: undecided, missed, incomplete, or wrong-object mask.
- `rejected`: unusable frame.

Annotate the bowl whenever visible in every behavior class. Green overlays are civet; orange overlays are food bowls.

# Finalize SAM 3 annotations and prepare the segmentation dataset
Run this notebook after 05_sam3_batch_annotation_from_processed.ipynb has processed the entire saved selection and a person has reviewed every overlay.

This notebook:

1. verifies that batch annotation is complete;
2. records human review decisions;
3. audits every accepted image/mask pair;
4. writes annotation summary reports;
5. converts accepted binary masks to YOLO Segmentation polygons;
6. preserves the video-level train/validation/test split from the original manifest.

It does not train a model yet.



# 1. Mount Drive and load annotation records

In [ ]:
from pathlib import Path
from datetime import datetime
import json, shutil
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/CMZprojectphase2')
ANNOTATION_ROOT = PROJECT_ROOT / 'data' / 'sam3_batch_annotation'
SELECTION_CSV = ANNOTATION_ROOT / 'annotation_selection.csv'
REVIEW_CSV = ANNOTATION_ROOT / 'review_status.csv'
PROPOSAL_DIR = ANNOTATION_ROOT / 'proposals'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'civet_bowl_yolo_segmentation'

for required in [SELECTION_CSV, REVIEW_CSV, PROPOSAL_DIR]:
    if not required.exists(): raise FileNotFoundError(required)
selection = pd.read_csv(SELECTION_CSV)
review = pd.read_csv(REVIEW_CSV)
if 'civet_status' not in review.columns and 'mask_status' in review.columns:
    review['civet_status'] = review['mask_status']
if 'bowl_status' not in review.columns:
    review['bowl_status'] = 'needs_review'
print('Selected:', len(selection), '| review rows:', len(review), '| output:', DATASET_ROOT)

## 2. Require complete batch processing

Do not finalize a partial first batch. If this cell fails, return to section 5 of notebook `05` and run the pending/batch cells repeatedly.

In [ ]:
selected_ids = set(selection['selection_id'].astype(str))
completed_ids = set(review['selection_id'].astype(str))
missing_civet = sorted(selected_ids - completed_ids)
missing_bowl = sorted(sid for sid in selected_ids if not (PROPOSAL_DIR / sid / 'bowl_metadata.json').exists())
print('Missing civet records:', len(missing_civet))
print('Missing bowl records:', len(missing_bowl))
if missing_civet: print('First missing civet IDs:', missing_civet[:20])
if missing_bowl: print('First missing bowl IDs:', missing_bowl[:20])
if missing_civet or missing_bowl:
    raise RuntimeError('Two-class annotation is incomplete. Finish the pending pass above.')
print('Civet and bowl proposal passes are complete.')

## 3. Record the human review

The user confirmed that all proposed masks were visually reviewed and correct, so `mask_proposed` rows are changed to `accepted`.

Rows where SAM returned no mask are **not** automatically accepted as negatives. Set `CONFIRM_NO_DETECTIONS_ARE_NEGATIVE = True` only after opening every such source image and confirming that no civet is visible.

In [ ]:
VALID_CIVET = {'accepted', 'corrected', 'negative', 'needs_review', 'rejected'}
VALID_BOWL = {'accepted', 'corrected', 'not_visible', 'needs_review', 'rejected'}
invalid_civet = sorted(set(review['civet_status'].dropna()) - VALID_CIVET)
invalid_bowl = sorted(set(review['bowl_status'].dropna()) - VALID_BOWL)
if invalid_civet or invalid_bowl:
    raise ValueError(f'Invalid statuses: civet={invalid_civet}, bowl={invalid_bowl}')
review.to_csv(REVIEW_CSV, index=False)
print('Saved review decisions:', REVIEW_CSV)
display(review[['civet_status', 'bowl_status']].value_counts(dropna=False).rename('images').to_frame())
if review.civet_status.eq('needs_review').any() or review.bowl_status.eq('needs_review').any():
    print('Review is not finished. Correct or confirm every needs_review item before export.')

## 4. Audit accepted masks

An accepted mask must exist, match the image dimensions, contain only 0/255, and contain foreground pixels. Any problem blocks dataset export.

In [ ]:
problems, audit_rows = [], []
class_specs = [
    ('civet', 'civet_status', 'mask.png', {'accepted', 'corrected'}),
    ('food_bowl', 'bowl_status', 'mask_bowl.png', {'accepted', 'corrected'}),
]
for row in review.itertuples():
    item_dir = PROPOSAL_DIR / str(row.selection_id)
    image_path = item_dir / 'image.jpg'
    if not image_path.exists():
        problems.append({'selection_id': row.selection_id, 'class_name': 'image', 'problem': 'missing image.jpg'})
        continue
    image = Image.open(image_path).convert('RGB')
    for class_name, status_column, filename, positive_statuses in class_specs:
        status = getattr(row, status_column)
        if status not in positive_statuses: continue
        mask_path = item_dir / filename
        if not mask_path.exists():
            problems.append({'selection_id': row.selection_id, 'class_name': class_name, 'problem': f'missing {filename}'})
            continue
        mask = np.asarray(Image.open(mask_path).convert('L'))
        values = set(np.unique(mask).tolist())
        messages = []
        if image.size != (mask.shape[1], mask.shape[0]): messages.append('image/mask size mismatch')
        if not values.issubset({0,255}): messages.append(f'invalid values {sorted(values)[:20]}')
        foreground = int((mask > 0).sum())
        if foreground == 0: messages.append('empty foreground')
        for message in messages:
            problems.append({'selection_id': row.selection_id, 'class_name': class_name, 'problem': message})
        audit_rows.append({'selection_id': row.selection_id, 'class_name': class_name, 'status': status,
                           'foreground_pixels': foreground, 'foreground_fraction': foreground/mask.size})
audit = pd.DataFrame(audit_rows)
problems_df = pd.DataFrame(problems, columns=['selection_id','class_name','problem'])
audit.to_csv(ANNOTATION_ROOT / 'accepted_mask_audit.csv', index=False)
problems_df.to_csv(ANNOTATION_ROOT / 'accepted_mask_problems.csv', index=False)
print('Audited masks:', len(audit), '| problems:', len(problems_df))
if len(problems_df):
    display(problems_df.head(30)); raise RuntimeError('Fix mask problems before export.')

In [ ]:
# Optional civet retry list. This does not affect the independent bowl pass.
retry_rows = review[(review['civet_status'] == 'needs_review') &
                    (review['civet_result'] == 'no_sam_detection')].copy()
print('Civet masks eligible for retry:', len(retry_rows))
display(retry_rows[['selection_id','behavior','lighting','civet_score']])

In [ ]:
# Retry is optional and requires a live GPU session with sam_processor loaded.
# A successful retry remains needs_review until a person checks the new overlay.
print('Use the retry loop only when GPU is available; otherwise correct these frames in CVAT.')

## 5. Generate annotation reports

In [ ]:
summary = (review.groupby(['behavior','lighting','civet_status','bowl_status'], dropna=False)
           .size().rename('images').reset_index())
summary.to_csv(ANNOTATION_ROOT / 'annotation_review_summary.csv', index=False)
display(summary)

civet_ok = review.civet_status.isin(['accepted','corrected'])
bowl_ok = review.bowl_status.isin(['accepted','corrected'])
report = f'''# CMZ SAM 3 Multi-class Pixel Annotation Report

- Selected frames: {len(selection)}
- Accepted/corrected civet masks: {int(civet_ok.sum())}
- Accepted/corrected food-bowl masks: {int(bowl_ok.sum())}
- Civet needs review: {int(review.civet_status.eq('needs_review').sum())}
- Bowl needs review: {int(review.bowl_status.eq('needs_review').sum())}

## Label definitions

- `civet`: all visible civet pixels.
- `food_bowl`: the visible feeding bowl or tray, excluding food, floor, shadow, and civet pixels.

Bowl labels are required whenever the bowl is visible in any behavior class. Eating remains a temporal behavior label; the presence of a bowl alone is not Eating.
'''
REPORT_PATH = ANNOTATION_ROOT / 'annotation_report.md'
REPORT_PATH.write_text(report, encoding='utf-8')
print(report)

## 6. Prepare YOLO Segmentation directories

Accepted/corrected masks become class `0` (`civet`). Human-confirmed negative images receive an empty label file. Rows still marked `needs_review` or `rejected` are excluded.

The original manifest split is preserved. YOLO calls validation `val`; test remains test.

In [ ]:
if 'split' not in selection.columns:
    raise ValueError('annotation_selection.csv has no split column.')

eligible = review.merge(selection[['selection_id','split']], on='selection_id', how='left', validate='one_to_one')
eligible['split'] = eligible['split'].replace({'validation':'val'}).fillna('train')
allowed_civet = {'accepted','corrected','negative'}
allowed_bowl = {'accepted','corrected','not_visible'}
final_decision = eligible.civet_status.isin(allowed_civet) & eligible.bowl_status.isin(allowed_bowl)
excluded = eligible[~final_decision].copy()
eligible = eligible[final_decision].copy()
excluded.to_csv(ANNOTATION_ROOT / 'excluded_needs_review.csv', index=False)
print('Eligible two-class images:', len(eligible))
print('Excluded because review is incomplete:', len(excluded))
display(eligible[['civet_status','bowl_status']].value_counts(dropna=False).rename('images').to_frame())
if eligible.empty:
    raise RuntimeError('No fully reviewed images are available.')
if DATASET_ROOT.exists():
    raise FileExistsError(f'{DATASET_ROOT} already exists. Rename or deliberately remove it before rebuilding.')
for split in ['train','val','test']:
    (DATASET_ROOT/'images'/split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT/'labels'/split).mkdir(parents=True, exist_ok=True)

## 7. Convert binary masks to YOLO polygons

The largest external contour is used as the single civet instance. The export records how much foreground area that contour retains; values below 0.98 are flagged because they may indicate disconnected mask fragments.

In [ ]:
import cv2

def mask_to_yolo_polygon(mask_array, epsilon_fraction=0.001):
    binary = (mask_array > 0).astype(np.uint8)*255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours: raise ValueError('No foreground contour')
    contour = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(contour, epsilon_fraction*cv2.arcLength(contour, True), True).reshape(-1,2)
    if len(approx) < 3: raise ValueError('Polygon has fewer than three points')
    h,w = binary.shape
    return np.column_stack([approx[:,0]/w, approx[:,1]/h]).clip(0,1)

export_rows, export_problems = [], []
for row in eligible.itertuples():
    item_dir = PROPOSAL_DIR / str(row.selection_id)
    image_target = DATASET_ROOT/'images'/row.split/f'{row.selection_id}.jpg'
    label_target = DATASET_ROOT/'labels'/row.split/f'{row.selection_id}.txt'
    shutil.copy2(item_dir/'image.jpg', image_target)
    label_lines, exported_classes = [], []
    specs = [
        (0, 'civet', row.civet_status, 'mask.png'),
        (1, 'food_bowl', row.bowl_status, 'mask_bowl.png'),
    ]
    for class_id, class_name, status, filename in specs:
        if status not in {'accepted','corrected'}: continue
        try:
            polygon = mask_to_yolo_polygon(np.asarray(Image.open(item_dir/filename).convert('L')))
            coordinates = ' '.join(f'{v:.6f}' for point in polygon for v in point)
            label_lines.append(f'{class_id} {coordinates}')
            exported_classes.append(class_name)
        except Exception as error:
            export_problems.append({'selection_id': row.selection_id, 'class_name': class_name, 'problem': str(error)})
    label_target.write_text(('\n'.join(label_lines)+'\n') if label_lines else '', encoding='utf-8')
    export_rows.append({'selection_id': row.selection_id, 'video_id': row.video_id,
                        'behavior': row.behavior, 'lighting': row.lighting, 'split': row.split,
                        'exported_classes': ','.join(exported_classes)})
export_manifest = pd.DataFrame(export_rows)
export_problem_df = pd.DataFrame(export_problems, columns=['selection_id','class_name','problem'])
export_manifest.to_csv(DATASET_ROOT/'dataset_manifest.csv', index=False)
export_problem_df.to_csv(DATASET_ROOT/'export_problems.csv', index=False)
print('Exported:', len(export_manifest), '| problems:', len(export_problem_df))
if len(export_problem_df): display(export_problem_df.head(30))

## 8. Write dataset configuration and final checks

In [ ]:
yaml_text = f'''path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: civet
  1: food_bowl
'''
(DATASET_ROOT/'dataset.yaml').write_text(yaml_text, encoding='utf-8')
checks = []
for split in ['train','val','test']:
    images = sorted((DATASET_ROOT/'images'/split).glob('*.jpg'))
    labels = sorted((DATASET_ROOT/'labels'/split).glob('*.txt'))
    checks.append({'split':split, 'images':len(images), 'labels':len(labels),
                   'pairs_match':{p.stem for p in images} == {p.stem for p in labels}})
checks = pd.DataFrame(checks)
display(checks)
if not checks.pairs_match.all(): raise RuntimeError('Image/label pairs do not match.')
print('Two-class YOLO dataset:', DATASET_ROOT)
print('Class 0=civet, class 1=food_bowl')

## Completion criteria

The dataset is ready only when both `civet_status` and `bowl_status` have no `needs_review` values, accepted/corrected masks pass the audit, every visible bowl is labeled across all behaviors, and the train/val/test image-label pairs match.